# RainGuard — Module 2: Exploratory Data Analysis (EDA)
**Indian Rainfall Forecasting and Heavy Rainfall Risk Analysis System**

**Input:** `data/processed/rainfall_cleaned.csv` (created by the Module 1 notebook).

**What this notebook does:** 13 charts (Matplotlib + Seaborn), each followed by a plain-language explanation, plus two data-quality discoveries that matter for modelling.

**Reading the numbers correctly**
* The data covers only **19 Aug – 30 Oct 2026** (73 days), the tail of the monsoon and the start of the post-monsoon season. "Total rainfall" below means the total over these 73 days, **not** a full year.
* Rainfall is measured in millimetres per day per district.
* Rankings by *total* rainfall are fair between full districts. For states, which have very different numbers of districts, we compare **mean daily rainfall per district-day** instead.

## Step 1 — Imports, paths and chart style
Colour rules used in every chart: one blue for a single measure, blue vs orange only when two measures are compared (actual vs normal), a light recessive grid, and no dual axes.

In [ ]:
import os, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

cwd = os.getcwd()
PROJECT_ROOT = os.path.abspath(os.path.join(cwd, "..")) if os.path.basename(cwd) == "notebooks" else cwd
CLEAN_PATH = os.path.join(PROJECT_ROOT, "data", "processed", "rainfall_cleaned.csv")
FIG_DIR = os.path.join(PROJECT_ROOT, "docs", "figures")
os.makedirs(FIG_DIR, exist_ok=True)
if not os.path.exists(CLEAN_PATH):
    raise FileNotFoundError("Run the Module 1 notebook first to create " + CLEAN_PATH)

# ---- chart style ----
BLUE, ORANGE = "#2a78d6", "#eb6834"
INK, INK2, GRID, SURFACE = "#0b0b0b", "#52514e", "#e4e3df", "#fcfcfb"
DIVERGING = LinearSegmentedColormap.from_list("blue_neutral_orange", [BLUE, "#f1f0ec", ORANGE])
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": "#b9b8b2", "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2,
    "text.color": INK, "axes.titleweight": "bold", "axes.titlesize": 12, "axes.titlelocation": "left",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.7, "axes.axisbelow": True,
    "font.size": 10, "legend.frameon": False,
})

def pretty(name):
    return name.title().replace("Nct Of", "NCT of").replace(" And ", " and ")

def save_show(name):
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, f"m2_{name}.png"), dpi=150)
    plt.show()

print("Cleaned file:", CLEAN_PATH)

## Step 2 — Load the cleaned data and build summary tables
We create two summary tables used by many charts:
* `dist_stats`: one row per district (total rainfall, mean, standard deviation, coefficient of variation, actual-to-normal ratio).
* `state_stats`: one row per state (number of districts, mean daily actual and normal rainfall).

**Note:** Nagapattinam (Tamil Nadu) has only 31 of 73 days, so its *totals* are not comparable with other districts. Charts that compare district totals exclude districts with missing days.

In [ ]:
date_cols = ["date", "cumulative_date", "week_start", "week_end", "month_start", "month_end"]
df = pd.read_csv(CLEAN_PATH, parse_dates=date_cols)

with open(os.path.join(PROJECT_ROOT, "models", "intensity_thresholds.json")) as f:
    INTENSITY_LABELS = json.load(f)["labels"]
df["rain_intensity"] = pd.Categorical(df["rain_intensity"], categories=INTENSITY_LABELS, ordered=True)

print("Shape:", df.shape, "| dates:", df["date"].min().date(), "->", df["date"].max().date())

dist_stats = (df.groupby(["state", "district"])
                .agg(total_actual=("daily_actual", "sum"), total_normal=("daily_normal", "sum"),
                     mean_daily=("daily_actual", "mean"), std_daily=("daily_actual", "std"),
                     days=("daily_actual", "size"))
                .reset_index())
dist_stats["cv"] = dist_stats["std_daily"] / dist_stats["mean_daily"].replace(0, np.nan)
dist_stats["actual_to_normal"] = dist_stats["total_actual"] / dist_stats["total_normal"].replace(0, np.nan)
FULL_DAYS = df["date"].nunique()
full = dist_stats[dist_stats["days"] == FULL_DAYS]      # districts with all 73 days

state_stats = (df.groupby("state")
                 .agg(districts=("district", "nunique"), mean_daily=("daily_actual", "mean"),
                      mean_normal=("daily_normal", "mean"))
                 .sort_values("mean_daily", ascending=False))
display(dist_stats.head())
display(state_stats.head())

## Chart 1 — Rainfall distribution
**What we are doing:** looking at how daily rainfall values are spread across all 53,102 district-days. The left panel uses a log scale on the vertical axis (otherwise the huge number of near-zero days hides everything else); the right panel zooms into rainy days up to 60 mm.

In [ ]:
x = df["daily_actual"]
rainy = x[x > 0]
print(f"Share of district-days with 0 mm : {(x == 0).mean() * 100:.1f}%")
print(f"Mean {x.mean():.2f} mm | median {x.median():.2f} mm | std {x.std():.2f} mm | max {x.max():.1f} mm")
print(f"Skewness {x.skew():.2f} | excess kurtosis {x.kurt():.2f}")
print(f"Rainy days above 60 mm (not shown in right panel): {(rainy > 60).sum()}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(x, bins=60, color=BLUE)
axes[0].set_yscale("log")
axes[0].set(xlabel="Daily rainfall (mm)", ylabel="District-days (log scale)", title="All district-days")
axes[1].hist(rainy, bins=60, range=(0, 60), color=BLUE)
axes[1].set(xlabel="Daily rainfall (mm), rainy days only, 0-60 mm", ylabel="District-days",
            title="Zoom on rainy days")
save_show("01_distribution")

**Findings.**
* About **55.6%** of district-days have exactly 0 mm, and the average day has only about 4.3 mm, while the median is 0.
* The distribution is extremely **right-skewed** (skewness ≈ 5.7) with very heavy tails (excess kurtosis ≈ 51): a few very wet days are far above everything else, with a maximum of 231.9 mm.
* Among rainy days, most are small; the count falls steadily as rainfall increases.

**Why it matters.** A model that predicts "no rain" would be right more than half the time, so accuracy-style numbers will look good without being useful. We need error measures in millimetres (MAE, RMSE), a comparison with a naive baseline, and extra attention to the rare heavy days.

## Chart 2 — Top 10 states by rainfall
**What we are doing:** ranking states by **mean daily rainfall per district** over the 73 days. We use the mean rather than the sum so that a state with many districts does not win just by being big.

In [ ]:
top_states = state_stats.head(10).iloc[::-1]
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh(top_states.index.str.title(), top_states["mean_daily"], color=BLUE, height=0.6)
for i, (v, n) in enumerate(zip(top_states["mean_daily"], top_states["districts"])):
    ax.text(v + 0.15, i, f"{v:.1f} mm  ({n} districts)", va="center", fontsize=9, color=INK2)
ax.set_xlim(0, top_states["mean_daily"].max() * 1.4)
ax.set(xlabel="Mean daily rainfall per district (mm)", title="Top 10 states by mean daily rainfall")
ax.grid(axis="y", visible=False)
save_show("02_top_states")
display(state_stats.head(10).round(2))

**Findings.**
* **Andaman & Nicobar** is the wettest by a wide margin (about 12.7 mm per district-day), followed by **Mizoram** (about 9.1) and **Odisha** (about 8.0).
* Uttarakhand, Kerala, Chhattisgarh, West Bengal, Sikkim, Jharkhand and Goa complete the top 10.
* Andaman & Nicobar (3 districts), Sikkim (2) and Goa (2) have very few districts, so their averages rest on very few places; Odisha's high average is based on 30 districts and is much more robust.

**Why it matters.** Location is clearly informative, so `state_enc` and `district_enc` are sensible model features. Remember that this is a ranking for *this* 73-day window, not an annual climatology.

## Chart 3 — Top 10 districts by rainfall
**What we are doing:** ranking districts by their **total** rainfall over the period (only districts with all 73 days are compared).

In [ ]:
top_d = full.nlargest(10, "total_actual").iloc[::-1]
labels = top_d["district"].str.title() + " (" + top_d["state"].str.title() + ")"
fig, ax = plt.subplots(figsize=(9, 4.8))
ax.barh(labels, top_d["total_actual"], color=BLUE, height=0.6)
for i, (a, nrm) in enumerate(zip(top_d["total_actual"], top_d["total_normal"])):
    ax.text(a + 15, i, f"{a:,.0f} mm  (normal {nrm:,.0f})", va="center", fontsize=9, color=INK2)
ax.set_xlim(0, top_d["total_actual"].max() * 1.35)
ax.set(xlabel="Total rainfall, 19 Aug - 30 Oct 2026 (mm)", title="Top 10 districts by total rainfall")
ax.grid(axis="y", visible=False)
save_show("03_top_districts")
display(top_d.iloc[::-1][["state", "district", "total_actual", "total_normal", "actual_to_normal"]].round(2))

**Findings.**
* **North & Middle Andaman** leads with about 1,491 mm, far ahead of the rest (Koraput about 964 mm, Nabarangapur about 922 mm).
* Odisha (3 districts) and Mizoram (2) appear repeatedly, matching Chart 2.
* **Banda (Uttar Pradesh)** is an unusual entry: its rainfall is more than three times its normal. Uttar Pradesh also has the most heavy-rainfall events of any state (Chart 11).
* Two entries, Balrampur (Chhattisgarh) and Balarampur (Uttar Pradesh), show *exactly* the same total. That is not a coincidence; it is explained in the next step.

## Data-quality discovery — identical districts and districts with no rain at all
Because two different districts had exactly the same total, we check whether any districts have **identical day-by-day rainfall series**, and which districts recorded 0 mm on every day.

In [ ]:
p = df.pivot_table(index="date", columns=["state", "district"], values="daily_actual")

series_map = {}
for col in p.columns:
    key = tuple(p[col].fillna(-1).round(1).values)
    series_map.setdefault(key, []).append(col)
dup_groups = [v for v in series_map.values() if len(v) > 1 and p[v[0]].fillna(0).sum() > 0]

print(f"Groups of districts with an IDENTICAL daily series (and some rain): {len(dup_groups)}")
print(f"Districts involved: {sum(len(g) for g in dup_groups)}\n")
for g in dup_groups:
    print(f"  {p[g[0]].sum():7.1f} mm total ->", " | ".join(f"{d.title()} ({s.title()})" for s, d in g))

zero_total = [c for c in p.columns if p[c].fillna(0).sum() == 0]
print("\nDistricts with 0 mm on every day:")
for s, d in zero_total:
    print("  ", d.title(), f"({s.title()})")

**Findings.**
* **7 groups (15 districts)** share exactly the same daily rainfall *and* the same daily normals. Two pairs are same-name districts in different states (Balrampur/Balarampur, Bijapur/Bijaapur); the other groups are different districts of the same state (five in Assam and Odisha). Real weather does not repeat to the decimal, so a plausible explanation is that IMD assigns one station's data to several districts. We cannot confirm that from the file.
* **4 districts** recorded 0 mm on all 73 days: Dibang Valley and Lower Dibang Valley (Arunachal Pradesh), Kargil (Ladakh) and North Delhi. A district with no rain at all for 73 days in a monsoon season may be a missing-data placeholder rather than real dryness, but the file does not tell us which.

**Our decision.** We do **not** delete or edit these records, because we cannot verify the truth and the project rules forbid inventing data. We **document them as a limitation**. They matter for modelling in two ways: duplicated series make some districts look "easier" than they are, and all-zero districts teach the model that some places never rain. In the viva you can present this as an honest data-quality finding.

## Chart 4 — Daily rainfall trend
**What we are doing:** averaging rainfall across all districts for each date and comparing it with the average normal for the same date.

In [ ]:
daily = df.groupby("date").agg(actual=("daily_actual", "mean"), normal=("daily_normal", "mean"))
peak_date, trough_date = daily["actual"].idxmax(), daily["actual"].idxmin()
print(f"Wettest day on average: {peak_date.date()} ({daily['actual'].max():.1f} mm)")
print(f"Driest day on average : {trough_date.date()} ({daily['actual'].min():.2f} mm)")
print(f"Correlation between the daily average actual and normal: {daily['actual'].corr(daily['normal']):.2f}")

fig, ax = plt.subplots(figsize=(11, 4.2))
ax.plot(daily.index, daily["actual"], color=BLUE, linewidth=2)
ax.plot(daily.index, daily["normal"], color=ORANGE, linewidth=2)
ax.text(daily.index[-1], daily["actual"].iloc[-1] + 0.3, " Actual", color=BLUE, va="bottom", fontweight="bold")
ax.text(daily.index[-1], daily["normal"].iloc[-1] - 0.3, " Normal", color=ORANGE, va="top", fontweight="bold")
ax.annotate(f"{peak_date:%d %b}: {daily['actual'].max():.1f} mm", (peak_date, daily["actual"].max()),
            xytext=(-95, -6), textcoords="offset points", fontsize=9, color=INK2,
            arrowprops=dict(arrowstyle="-", color=INK2, linewidth=0.8))
ax.set(ylabel="Mean rainfall across districts (mm/day)", title="Daily rainfall across India: actual vs normal")
ax.set_xlim(daily.index[0], daily.index[-1] + pd.Timedelta(days=7))
save_show("04_daily_trend")

**Findings.**
* The **normal** (orange) falls steadily from about 8.7 mm in late August to under 2 mm at the end of October: the monsoon is withdrawing.
* **Actual** rainfall (blue) is much noisier. It tracks the normal only loosely (correlation about 0.59), with a large spike on **26 September** (about 13.8 mm average over all districts) and a very dry spell around **13 October** (about 0.2 mm).
* Overall, actual rainfall ran below normal (about 83% of it, see Chart 8), but it exceeded the normal on a few occasions, notably the late-September burst and the very end of October.

**Why it matters.** Rain arrives in short, widespread bursts rather than a smooth curve, and the overall level drops through the window. A model trained on the early, wetter part and tested on the later, drier part will face a different regime; we must keep this in mind when interpreting test errors in Module 5.

## Chart 5 — State-wise rainfall comparison
**What we are doing:** a box plot per state of its districts' mean daily rainfall. The box shows the middle 50% of districts; the line is the median. States are ordered by median.

In [ ]:
order = dist_stats.groupby("state")["mean_daily"].median().sort_values(ascending=False).index
fig, ax = plt.subplots(figsize=(9, 10))
sns.boxplot(data=dist_stats, y="state", x="mean_daily", order=order, color=BLUE, saturation=0.8,
            width=0.6, fliersize=2, linewidth=0.8, ax=ax)
ax.set_yticklabels([pretty(t.get_text()) for t in ax.get_yticklabels()])
ax.set(xlabel="Mean daily rainfall per district (mm)", ylabel="", title="Rainfall spread between districts, by state")
ax.grid(axis="y", visible=False)
save_show("05_state_boxplot")

spread = (dist_stats.groupby("state")["mean_daily"]
          .agg(districts="size", median="median", iqr=lambda s: s.quantile(.75) - s.quantile(.25)))
print("States with the widest district-to-district spread (at least 5 districts):")
display(spread[spread["districts"] >= 5].sort_values("iqr", ascending=False).head(5).round(2))

**Findings.**
* States differ strongly in their typical rainfall. By mean daily rainfall (Chart 2 and the state table), Andaman & Nicobar, Mizoram, Odisha, Uttarakhand and Kerala are wettest, while Ladakh, Punjab, Rajasthan, Haryana and Gujarat are driest.
* Some states are internally very mixed. Andhra Pradesh has the widest spread between its districts (box width about 5.1 mm/day), followed by Arunachal Pradesh, Madhya Pradesh, Odisha and Chhattisgarh (about 3.6 to 3.9).

**Why it matters.** State alone is not enough to describe rainfall: districts in the same state can behave very differently, so we include a **district** encoding too, and we will model each district from its own recent history.

## Chart 6 — District-wise rainfall distribution
**What we are doing:** a histogram of the 73-day total rainfall for each of the 727 districts with full records.

In [ ]:
t = full["total_actual"]
print(t.describe().round(1).to_frame("total mm"))

fig, ax = plt.subplots(figsize=(9, 4.2))
ax.hist(t, bins=40, color=BLUE)
ax.axvline(t.median(), color=INK2, linestyle="--", linewidth=1.2)
ax.text(t.median() + 12, ax.get_ylim()[1] * 0.92, f"median {t.median():.0f} mm", color=INK2, fontsize=9)
ax.set(xlabel="Total rainfall per district, 19 Aug - 30 Oct 2026 (mm)", ylabel="Number of districts",
       title="How much rain did a typical district get?")
save_show("06_district_totals")

**Findings.**
* A typical district received about **290 mm** over the 73 days (mean about 312 mm), but the spread is wide: the middle 50% of districts lie between roughly 147 mm and 436 mm.
* The right tail is long: four districts received more than 900 mm, the top one about 1,491 mm.
* A few districts sit at 0 mm (the all-zero districts found earlier).

**Why it matters.** Districts live on very different rainfall scales. A single national model must cope with this, which is another reason to give it the district's own recent rainfall as input.

## Chart 7 — Monthly rainfall patterns within the available period
**What we are doing:** comparing August, September and October. **August only has 13 days (19–31 Aug)**, so it is a partial month.

In [ ]:
monthly = (df.groupby("month")
             .agg(days=("date", "nunique"), mean_daily=("daily_actual", "mean"),
                  rainy_share=("daily_actual", lambda s: (s > 0).mean() * 100),
                  heavy_events=("daily_actual", lambda s: int((s >= 64.5).sum()))))
display(monthly.round(2))

names = {8: "Aug (19-31)", 9: "Sep", 10: "Oct"}
lab = [f"{names[m]}\n{monthly.loc[m, 'days']} days" for m in monthly.index]
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].bar(lab, monthly["mean_daily"], color=BLUE, width=0.55)
axes[1].bar(lab, monthly["rainy_share"], color=BLUE, width=0.55)
for ax, vals, fmt in [(axes[0], monthly["mean_daily"], "{:.1f} mm"), (axes[1], monthly["rainy_share"], "{:.0f}%")]:
    for i, v in enumerate(vals):
        ax.text(i, v, fmt.format(v), ha="center", va="bottom", fontsize=9, color=INK2)
    ax.grid(axis="x", visible=False)
axes[0].set(ylabel="Mean daily rainfall per district (mm)", title="Average rainfall")
axes[1].set(ylabel="Share of district-days with rain (%)", title="How often it rained")
save_show("07_monthly_pattern")

**Findings.**
* Mean daily rainfall drops from about **6.5 mm (late Aug)** to **5.6 mm (Sep)** and then sharply to **2.0 mm (Oct)**.
* The share of district-days with any rain falls from about 69% to 56% to 22%.
* Heavy-or-above events (≥ 64.5 mm) occurred 84 times in the 13 days of August, 229 times in September and 47 times in October.

**Why it matters.** The data shows a strong seasonal decline within only 73 days, but we have **no full year**, so we cannot claim to learn seasonality. The model's `month` feature only has three values here, and we will be honest about that in the limitations.

## Chart 8 — Actual versus normal rainfall
**What we are doing:** each dot is a district. Its position shows its 73-day total actual rainfall against its 73-day total normal. Dots on the grey line received exactly the normal amount; dots below it were drier than normal, dots above it wetter.

In [ ]:
national_ratio = full["total_actual"].sum() / full["total_normal"].sum()
above = (full["actual_to_normal"] > 1).mean() * 100
print(f"All districts together: actual = {national_ratio * 100:.0f}% of normal")
print(f"Median district: {full['actual_to_normal'].median() * 100:.0f}% of normal")
print(f"Districts above normal: {above:.0f}%")

fig, ax = plt.subplots(figsize=(6.5, 6))
ax.scatter(full["total_normal"], full["total_actual"], s=14, color=BLUE, alpha=0.5, linewidths=0)
lim = max(full["total_normal"].max(), full["total_actual"].max()) * 1.05
ax.plot([0, lim], [0, lim], color=INK2, linewidth=1.2, linestyle="--")
ax.text(lim * 0.62, lim * 0.68, "actual = normal", color=INK2, rotation=40, fontsize=9)
for _, r in full.nlargest(3, "actual_to_normal").iterrows():
    ax.annotate(r["district"].title(), (r["total_normal"], r["total_actual"]), xytext=(6, 4),
                textcoords="offset points", fontsize=8, color=INK2)
ax.set(xlim=(0, lim), ylim=(0, lim), xlabel="Total normal rainfall (mm)", ylabel="Total actual rainfall (mm)",
       title="Actual vs normal rainfall by district")
save_show("08_actual_vs_normal")
display(full.nlargest(5, "actual_to_normal")[["state", "district", "total_actual", "total_normal", "actual_to_normal"]].round(2))
display(full.nsmallest(5, "actual_to_normal")[["state", "district", "total_actual", "total_normal", "actual_to_normal"]].round(2))

**Findings.**
* Overall, districts received about **83% of their normal** rainfall in this window, and only about **34% of districts** were above normal. It was a drier-than-normal period for most of India.
* Most dots lie **below** the dashed line. The few far above it (labelled, e.g. Gautambudhnagar, Mahoba and Banda) are mostly in Uttar Pradesh, at roughly three times their normal.
* On the far right, four districts had very high normals (1,300 to 1,700 mm) but received only about a fifth to a quarter of that (290 to 365 mm): East Jaintia Hills, West Jaintia Hills and East Khasi Hills in Meghalaya, and Tamenglong in Manipur.
* The dots at the bottom edge are the districts with 0 mm. Overall there is a lot of scatter around the line.

**Why it matters.** The "normal" is a long-term average, so it says little about what will happen tomorrow. That supports building forecasts from *recent actual* rainfall (lags) rather than from normals.

## Chart 9 — Rainfall category distribution
**What we are doing:** two different category systems side by side.
* **Left:** IMD's `daily_category` (how the day compares with normal): NR = no rain, LD = large deficient, D = deficient, N = normal, E = excess, LE = large excess, ND = no data.
* **Right:** our `rain_intensity` (how many millimetres fell), which is what the classifier will predict.

In [ ]:
imd_order = ["NR", "LD", "D", "N", "E", "LE", "ND"]
imd = df["daily_category"].value_counts().reindex(imd_order)
inten = df["rain_intensity"].value_counts().reindex(INTENSITY_LABELS)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.3))
axes[0].bar(imd.index, imd.values, color=BLUE, width=0.6)
axes[1].bar(inten.index, inten.values, color=BLUE, width=0.6)
axes[1].set_yscale("log")
for ax, s in [(axes[0], imd), (axes[1], inten)]:
    for i, v in enumerate(s.values):
        ax.text(i, v, f"{v:,}", ha="center", va="bottom", fontsize=8.5, color=INK2)
    ax.grid(axis="x", visible=False)
axes[0].set(ylabel="District-days", title="IMD daily category (vs normal)")
axes[1].set(ylabel="District-days (log scale)", title="Rainfall intensity (mm, IMD thresholds)")
axes[1].tick_params(axis="x", rotation=20)
save_show("09_categories")
display(pd.DataFrame({"rows": inten, "percent": (inten / len(df) * 100).round(3)}))

**Findings.**
* IMD's categories are dominated by **NR** (no rain, about 53% of days), followed by LD and LE. Because they measure departure from normal, a "large excess" day can still be tiny in millimetres.
* Our intensity classes are far more imbalanced: **73.6%** are No/Very Light, **18.1%** Light, **7.6%** Moderate, and only **0.57%** Heavy (302 records), **0.10%** Very Heavy (52) and **0.01%** Extremely Heavy (6).

**Why it matters.** The classifier's hardest and most important classes are the rarest. We need class weighting, a report of per-class recall, and a decision on merging the rarest classes (Module 6).

## Chart 10 — Correlation heatmap
**What we are doing:** measuring the linear relationship between the numeric rainfall columns. Values near +1 or −1 mean a strong relationship; near 0 means weak. **This chart is for exploration only**: the weekly, monthly, cumulative and category columns will not be model inputs (Module 1, Step 12).

In [ ]:
cols = ["daily_actual", "daily_normal", "weekly_actual", "weekly_normal", "monthly_actual",
        "monthly_normal", "cumulative_actual", "cumulative_normal", "daily_category_code"]
corr = df[cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap=DIVERGING, vmin=-1, vmax=1, center=0,
            linewidths=1, linecolor=SURFACE, annot_kws={"size": 8.5}, cbar_kws={"shrink": 0.8}, ax=ax)
ax.set_title("Correlation between rainfall measures")
ax.grid(False)
save_show("10_correlation")
print("Daily actual vs the other columns:")
display(corr["daily_actual"].drop("daily_actual").sort_values(ascending=False).round(2).to_frame("correlation"))

**Findings.**
* `daily_actual` is strongly related only to `daily_category_code` (about 0.67), which is expected because IMD derived that category from the same-day rainfall.
* Its relationship with the weekly, monthly and cumulative totals is weak (about 0.2 to 0.35) and with the same day's normal only about 0.19.
* The strongest correlations are between each *actual* and its own *normal* at the longer windows (cumulative about 0.87, monthly about 0.77), which simply says that wet places stay wet over a season.

**Why it matters.** No simple column gives away tomorrow's rainfall linearly. This is why we build lag and rolling features from each district's own past rainfall, and why we expect modest forecasting skill, which we will report honestly.

## Chart 11 — Heavy rainfall observations
**What we are doing:** looking at "heavy or above" events, meaning at least 64.5 mm in a day (IMD Heavy, Very Heavy and Extremely Heavy). Left: which states had the most such district-days. Right: how many occurred on each date.

In [ ]:
HEAVY_MM = 64.5
heavy = df[df["daily_actual"] >= HEAVY_MM]
print(f"Heavy-or-above district-days: {len(heavy)} ({len(heavy) / len(df) * 100:.2f}% of all records)")

by_state = heavy.groupby("state").size().sort_values(ascending=False).head(10).iloc[::-1]
by_date = heavy.groupby("date").size().reindex(daily.index, fill_value=0)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), gridspec_kw={"width_ratios": [1, 1.5]})
axes[0].barh(by_state.index.str.title(), by_state.values, color=BLUE, height=0.6)
for i, v in enumerate(by_state.values):
    axes[0].text(v + 0.8, i, str(v), va="center", fontsize=9, color=INK2)
axes[0].set(xlabel="Heavy-or-above district-days", title="Top 10 states")
axes[0].grid(axis="y", visible=False)
axes[1].bar(by_date.index, by_date.values, color=BLUE, width=0.8)
axes[1].set(ylabel="Heavy-or-above districts on the day", title="Heavy rainfall events by date")
save_show("11_heavy_rainfall")

top_days = by_date.sort_values(ascending=False).head(4)
print("Busiest days:", {d.strftime("%d %b"): int(n) for d, n in top_days.items()})
print(f"Those 4 days hold {top_days.sum() / len(heavy) * 100:.0f}% of all heavy-or-above events")

**Findings.**
* There are **360** heavy-or-above district-days (about 0.68% of records).
* **Uttar Pradesh** has the most (74), then Andhra Pradesh (40), Chhattisgarh (40), Odisha (36) and Madhya Pradesh (32).
* The events are strongly **clustered in time**: **24–27 September** holds 140 of the 360 events (about 39%), peaking at 47 districts on 26 September. A smaller cluster appears on 28 October (12 districts), and a few other days reach 11–13.

**Why it matters.** Heavy rain here comes from a handful of large weather systems affecting many neighbouring districts at once, so the heavy-rain "samples" are not independent. With so few events, a chronological split may leave very few heavy days in the test set (checked in the last step below).

## Chart 12 — Rainfall variation across districts
**What we are doing:** measuring how erratic each district's daily rainfall is using the **coefficient of variation (CV)** = standard deviation ÷ mean. A high CV means rain comes in rare, intense bursts; a low CV means more even rainfall. Districts with zero rainfall (CV undefined) are left out.

In [ ]:
cv = full["cv"].dropna()
print(cv.describe().round(2).to_frame("CV"))

fig, ax = plt.subplots(figsize=(9, 4.2))
ax.hist(cv, bins=45, color=BLUE)
ax.axvline(cv.median(), color=INK2, linestyle="--", linewidth=1.2)
ax.text(cv.median() + 0.08, ax.get_ylim()[1] * 0.92, f"median {cv.median():.2f}", color=INK2, fontsize=9)
ax.set(xlabel="Coefficient of variation of daily rainfall (std / mean)", ylabel="Number of districts",
       title="How erratic is each district's rainfall?")
save_show("12_variability")

print("\nMost erratic districts (rain concentrated in very few days):")
display(full.nlargest(5, "cv")[["state", "district", "total_actual", "mean_daily", "std_daily", "cv"]].round(2))
print("Steadiest districts:")
display(full.nsmallest(5, "cv")[["state", "district", "total_actual", "mean_daily", "std_daily", "cv"]].round(2))

**Findings.**
* The **median CV is about 2.3**: for a typical district the day-to-day standard deviation is more than twice the average rainfall. Rainfall is bursty almost everywhere.
* The most erratic districts have very low totals (for example Sahebganj in Jharkhand with only about 19 mm in total, CV about 8.5): a few drops on a few days, then nothing.
* The steadiest districts (lowest CV, about 1.0 to 1.3) are wet ones such as Ri Bhoi (Meghalaya), South Andaman, Sindhudurg and Raigad (Maharashtra) and Aizawl (Mizoram).

**Why it matters.** The high variability explains why daily rainfall is hard to forecast and why an error of a few millimetres can be large relative to the average.

## Chart 13 (bonus) — Where and when did it rain? State × week heatmap
**What we are doing:** one picture of both space and time. Each cell is the average daily rainfall in a state during one 7-day period (the last column covers only 3 days). Darker blue means wetter. States are sorted from wettest to driest overall.

In [ ]:
df["week_no"] = (df["date"] - df["date"].min()).dt.days // 7
week_label = df.groupby("week_no")["date"].min().dt.strftime("%d %b")
heat = df.pivot_table(index="state", columns="week_no", values="daily_actual", aggfunc="mean")
heat = heat.loc[state_stats.index]
heat.columns = [week_label[w] for w in heat.columns]

fig, ax = plt.subplots(figsize=(11, 10))
sns.heatmap(heat, cmap="Blues", linewidths=0.6, linecolor=SURFACE,
            cbar_kws={"label": "Mean daily rainfall (mm)", "shrink": 0.7}, ax=ax)
ax.set_yticklabels([pretty(t.get_text()) for t in ax.get_yticklabels()], rotation=0)
ax.set(xlabel="Week starting", ylabel="", title="Mean daily rainfall by state and week")
ax.grid(False)
save_show("13_state_week_heatmap")

**Findings.**
* Most northern and central states are darkest in the **first weeks** and fade to near-white by October, the monsoon withdrawal again.
* The vertical dark band in the week of **23 Sep** across the eastern and central states (Odisha, Chhattisgarh, Jharkhand, West Bengal, Sikkim, Uttar Pradesh, Andhra Pradesh) is the late-September event seen in Charts 4 and 11.
* The darkest cells in the whole chart are **Odisha and Andhra Pradesh in the final 3 days** (28 to 30 Oct). In contrast, the southern states and islands (Tamil Nadu, Kerala, Puducherry, Lakshadweep, Andaman & Nicobar) are wettest in mid to late October, when most other states are dry.

**Why it matters.** Rain in one district is linked to rain in nearby districts and nearby days, so models that look at a district's last few days have real information to use.

## Heads-up for modelling — what a chronological split would look like
In Modules 5 and 6 we must split **by time** (train on earlier dates, test on later dates). This cell only *illustrates* what happens if the last 15 days become the test set; the real split is decided in Module 5.

In [ ]:
dates = np.sort(df["date"].unique())
cut = dates[-15]
test_part = df[df["date"] >= cut]
train_part = df[df["date"] < cut]
print(f"Illustration: test period starts {pd.Timestamp(cut).date()} ({len(dates[dates >= cut])} days)\n")

summary = pd.DataFrame({
    "rows": [len(train_part), len(test_part)],
    "mean daily rainfall (mm)": [train_part["daily_actual"].mean(), test_part["daily_actual"].mean()],
    "rows >= 64.5 mm (heavy+)": [(train_part["daily_actual"] >= 64.5).sum(), (test_part["daily_actual"] >= 64.5).sum()],
    "rows >= 115.6 mm (very heavy+)": [(train_part["daily_actual"] >= 115.6).sum(), (test_part["daily_actual"] >= 115.6).sum()],
    "rows >= 204.5 mm (extremely heavy)": [(train_part["daily_actual"] >= 204.5).sum(), (test_part["daily_actual"] >= 204.5).sum()],
}, index=["train (earlier)", "test (last 15 days)"]).round(2)
display(summary)

**What this tells us.**
* The later period is drier (about 2.6 mm/day against 4.7 mm/day) and has far fewer heavy-or-above rows (38 against 322; a rate of about 0.35% against 0.76%).
* The rarest class is split awkwardly: only **2** Extremely Heavy rows would be in training, while **4** of the 6 fall in the test period (the 28–29 Oct event). The test period also has only 9 Very-Heavy-or-above rows.
* A model cannot learn a class it has barely seen, and test scores for the rarest classes will rest on a handful of rows, so they cannot be judged reliably.

In Module 6 we will report this limitation openly and decide how to group the rare classes (for example a combined *Heavy or above* class for the headline result).

## Module 2 summary
| # | Chart | Key finding |
|---|---|---|
| 1 | Distribution | 55.6% zero-rain days; extreme right skew (about 5.7) |
| 2 | Top states | Andaman & Nicobar, Mizoram, Odisha lead per district-day |
| 3 | Top districts | North & Middle Andaman about 1,491 mm; Banda is 3x its normal |
| 4 | Daily trend | Normal declines through the window; huge burst on 26 Sep |
| 5 | State comparison | Big differences between states and inside some states |
| 6 | District totals | Median about 290 mm, long right tail |
| 7 | Monthly pattern | 6.5 → 5.6 → 2.0 mm/day (Aug partial, Sep, Oct) |
| 8 | Actual vs normal | About 83% of normal; about 34% of districts above normal |
| 9 | Categories | Heavy classes are 0.57% / 0.10% / 0.01% of records |
| 10 | Correlation | No strong linear predictor of daily rainfall |
| 11 | Heavy rainfall | 360 events; 39% fall on 24–27 Sep; Uttar Pradesh most |
| 12 | Variability | Median CV about 2.3: rain is bursty |
| 13 | State × week | Clear seasonal fade and a late-September band |

**Data-quality findings to report as limitations:** 7 groups (15 districts) with identical rainfall series; 4 districts with 0 mm on all 73 days; Nagapattinam has only 31 days.

### Viva questions for Module 2
**Q1. Why compare states by mean daily rainfall instead of total rainfall?**
States have very different numbers of districts, so a total would reward size. The mean per district-day is comparable.

**Q2. Why use a log scale for the distribution and intensity charts?**
Most days are near zero and a few are extreme; on a linear axis the rare large values would be invisible.

**Q3. Why does the correlation heatmap not mean we can use those columns as features?**
The weekly, monthly, cumulative and category columns are overlapping aggregates of the same rainfall, and including them risks leakage; the heatmap is only for understanding.

**Q4. What does a high coefficient of variation tell you?**
That a district's rainfall comes in rare intense bursts instead of steadily, which makes forecasting harder.

**Q5. What is the most important problem for modelling that EDA revealed?**
Heavy-rain events are extremely rare and clustered on a few dates, so the models will have little data on exactly the cases we care most about, and a chronological test set will contain very few of them.

*Next: Module 3 — Statistical Analysis.*